# Notebook merger

Notebook ini menggabungkan empat notebook proyek menjadi satu file `.ipynb` baru di folder `IDSC`.

Jalankan cell code di bawah untuk membuat file hasil gabungan.

In [1]:
from copy import deepcopy
from pathlib import Path
import json
import uuid

BASE_DIR = Path.cwd()

NOTEBOOKS = [
    ("bigP3BCI", BASE_DIR / "bigP3BCI" / "idsc-dawnbringer-bigp3bci.ipynb"),
    ("Brugada", BASE_DIR / "Brugada" / "idsc-dawnbringer-brugada.ipynb"),
    ("Hillel Yaffe Glaucoma", BASE_DIR / "Hillel Yaffe Glaucoma" / "idsc-dawnbringer_Glaucoma.ipynb"),
    ("SPECT", BASE_DIR / "SPECT" / "notebooks" / "HeartSPECT_FullProject.ipynb"),
]

OUTPUT_PATH = BASE_DIR / "IDSC_merged.ipynb"


def load_notebook(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def ensure_list_source(source):
    if isinstance(source, list):
        return source
    if isinstance(source, str):
        return source.splitlines(keepends=True)
    return []


def make_markdown_cell(text: str) -> dict:
    return {
        "cell_type": "markdown",
        "metadata": {},
        "source": text.splitlines(keepends=True),
        "id": uuid.uuid4().hex[:8],
    }


merged_cells = [
    make_markdown_cell(
        "# IDSC merged notebook\n\n"
        "File ini dibuat otomatis dari empat notebook sumber."
    )
]

metadata = {
    "kernelspec": {
        "display_name": "Python 3",
        "language": "python",
        "name": "python3",
    },
    "language_info": {
        "name": "python",
    },
}

missing_paths = []

for section_name, notebook_path in NOTEBOOKS:
    if not notebook_path.exists():
        missing_paths.append(str(notebook_path))
        continue

    notebook_data = load_notebook(notebook_path)

    merged_cells.append(
        make_markdown_cell(
            f"\\n---\\n\\n# Section: {section_name}\\n\\nSource: `{notebook_path.relative_to(BASE_DIR)}`\\n"
        )
    )

    for cell in notebook_data.get("cells", []):
        copied_cell = deepcopy(cell)
        copied_cell["id"] = uuid.uuid4().hex[:8]
        copied_cell["source"] = ensure_list_source(copied_cell.get("source", []))
        merged_cells.append(copied_cell)

    if notebook_data.get("metadata") and metadata["language_info"] == {"name": "python"}:
        metadata = deepcopy(notebook_data["metadata"])

if missing_paths:
    raise FileNotFoundError("Notebook tidak ditemukan:\n- " + "\n- ".join(missing_paths))

merged_notebook = {
    "cells": merged_cells,
    "metadata": metadata,
    "nbformat": 4,
    "nbformat_minor": 5,
}

with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(merged_notebook, f, ensure_ascii=False, indent=2)

print(f"Merged {len(NOTEBOOKS)} notebooks into: {OUTPUT_PATH}")
print(f"Total cells written: {len(merged_cells)}")


Merged 4 notebooks into: c:\Patrick\coding\IDSC\IDSC_merged.ipynb
Total cells written: 332
